# Stock Truth — Stock-Laya one-click training

This notebook fine-tunes Laya specifically on Stock Truth's historical stock states. It reuses the existing Q-State feature schema, keeps calibration and final test periods chronological, and never trains on the untouched test split.

**Kaggle settings:** Accelerator = **GPU T4 x2**, Internet = **On**. An optional Kaggle secret named `HF_TOKEN` with Hugging Face write permission uploads a passing candidate to `Smit1105/stock-laya-qstate`.


In [ ]:
import os, subprocess, torch
print('CUDA:', torch.cuda.is_available(), 'GPUs:', torch.cuda.device_count())
for i in range(torch.cuda.device_count()): print(i, torch.cuda.get_device_name(i))
assert torch.cuda.device_count() >= 2, 'Select Kaggle GPU T4 x2 before running.'


In [ ]:
!rm -rf /kaggle/working/stock-truth-v2 /kaggle/working/stock-truth-data
!git clone --depth 1 https://github.com/samin110597-create/stock-truth-v2.git /kaggle/working/stock-truth-v2
!git clone --depth 1 --branch data-snapshots https://github.com/samin110597-create/stock-truth-v2.git /kaggle/working/stock-truth-data
!mkdir -p /kaggle/working/stock-truth-v2/data
!cp -r /kaggle/working/stock-truth-data/raw /kaggle/working/stock-truth-v2/data/ 2>/dev/null || true
!cp -r /kaggle/working/stock-truth-data/quant /kaggle/working/stock-truth-v2/data/ 2>/dev/null || true
!pip install -q -U 'laya>=0.3.20' 'transformers>=5.0.0' 'huggingface_hub>=1.0.0' safetensors scikit-learn numpy


In [ ]:
%cd /kaggle/working/stock-truth-v2
!python scripts/build_laya_stock_dataset.py --min-cases 500
!python scripts/preprocess_stock_laya.py
import json
print(json.load(open('data/laya/report.json')))
print(json.load(open('data/laya/preprocessed/preprocess-report.json')))


In [ ]:
from huggingface_hub import snapshot_download
model_dir=snapshot_download('convaiinnovations/laya')
open('/kaggle/working/laya_model_dir.txt','w').write(model_dir)
print('Base checkpoint:', model_dir)


In [ ]:
MODEL_DIR=open('/kaggle/working/laya_model_dir.txt').read().strip()
OUT='/kaggle/working/stock-laya-qstate'
!torchrun --nproc_per_node=2 scripts/train_stock_laya_ddp.py "$MODEL_DIR" "$OUT" data/laya/preprocessed


In [ ]:
!python scripts/evaluate_stock_laya.py --model /kaggle/working/stock-laya-qstate --test data/laya/test.jsonl --out /kaggle/working/stock-laya-evaluation.json --device cuda
import json
report=json.load(open('/kaggle/working/stock-laya-evaluation.json'))
print(json.dumps(report,indent=2))


In [ ]:
# Optional automatic Hub upload. The notebook does not expose or print your token.
import os, json
from huggingface_hub import HfApi
token=None
try:
    from kaggle_secrets import UserSecretsClient
    token=UserSecretsClient().get_secret('HF_TOKEN')
except Exception as e:
    print('HF_TOKEN not configured; trained model remains in Kaggle output.')
if token:
    api=HfApi(token=token)
    repo_id='Smit1105/stock-laya-qstate'
    api.create_repo(repo_id,repo_type='model',private=True,exist_ok=True)
    api.upload_folder(repo_id=repo_id,repo_type='model',folder_path='/kaggle/working/stock-laya-qstate')
    api.upload_file(repo_id=repo_id,repo_type='model',path_or_fileobj='/kaggle/working/stock-laya-evaluation.json',path_in_repo='stock-laya-evaluation.json')
    print('Uploaded:',repo_id)
